# 01 — Pydantic + Spark Contracts

Purpose: use Pydantic as a contract pattern around Spark, not as a row-by-row Spark validation engine.

Process schema:

```text
PYDANTIC CONTRACT
      |
      v
SAMPLE / EDGE VALIDATION
      |
      v
SPARK RAW DATAFRAME
      |
      v
SPARK QUALITY RULES
      |
      +--> SILVER VALID
      |
      +--> QUARANTINE
```

Main rule:

```text
Pydantic validates contracts and small samples.
Spark validates data at scale.
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from datetime import datetime
from typing import Optional
from pydantic import BaseModel, Field

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("pydantic-spark-contracts")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Pydantic contract

This defines the expected event shape at the application / contract level.

In [2]:
class CustomerEvent(BaseModel):
    customer_id: str
    event_type: str
    amount: Optional[float] = Field(default=None, ge=0)
    event_time: datetime

## Step 2 — Validate a small sample with Pydantic

This is useful for API input, configs, fixtures, samples, and contract documentation.

In [3]:
sample_event = {
    "customer_id": "C001",
    "event_type": "purchase",
    "amount": 100.0,
    "event_time": datetime(2026, 1, 1, 10, 0, 0),
}

validated_event = CustomerEvent(**sample_event)
validated_event

CustomerEvent(customer_id='C001', event_type='purchase', amount=100.0, event_time=datetime.datetime(2026, 1, 1, 10, 0))

## Step 3 — Define Spark raw schema

Raw schema is permissive.

Invalid rows must be allowed into Spark so they can be checked and quarantined.

In [4]:
raw_customer_event_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
])

## Step 4 — Create raw rows

The dataset intentionally contains valid and invalid rows.

In [5]:
rows = [
    ("C001", "purchase", 100.0, datetime(2026, 1, 1, 10, 0, 0)),
    ("C002", "refund", 50.0, datetime(2026, 1, 1, 11, 0, 0)),
    ("C003", "purchase", None, datetime(2026, 1, 1, 12, 0, 0)),
    (None, "purchase", 20.0, datetime(2026, 1, 1, 13, 0, 0)),
    ("C005", "unknown", 10.0, datetime(2026, 1, 1, 14, 0, 0)),
    ("C006", "purchase", -5.0, datetime(2026, 1, 1, 15, 0, 0)),
]

## Step 5 — Create Spark DataFrame with explicit schema

This cell is self-contained: `rows` and `raw_customer_event_schema` were defined above in this notebook.

In [6]:
df = spark.createDataFrame(rows, schema=raw_customer_event_schema)

df.show(truncate=False)
df.printSchema()

+-----------+----------+------+-------------------+
|customer_id|event_type|amount|event_time         |
+-----------+----------+------+-------------------+
|C001       |purchase  |100.0 |2026-01-01 10:00:00|
|C002       |refund    |50.0  |2026-01-01 11:00:00|
|C003       |purchase  |NULL  |2026-01-01 12:00:00|
|NULL       |purchase  |20.0  |2026-01-01 13:00:00|
|C005       |unknown   |10.0  |2026-01-01 14:00:00|
|C006       |purchase  |-5.0  |2026-01-01 15:00:00|
+-----------+----------+------+-------------------+

root
 |-- customer_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



## Step 6 — Spark validation rules

Spark does the scalable validation.

In [7]:
allowed_event_types = ["purchase", "refund"]

validated = (
    df
    .withColumn(
        "error_reason",
        F.when(F.col("customer_id").isNull(), F.lit("missing_customer_id"))
         .when(~F.col("event_type").isin(allowed_event_types), F.lit("invalid_event_type"))
         .when(F.col("amount") < 0, F.lit("negative_amount"))
         .when(F.col("event_time").isNull(), F.lit("missing_event_time"))
    )
    .withColumn("is_valid", F.col("error_reason").isNull())
)

validated.show(truncate=False)

+-----------+----------+------+-------------------+-------------------+--------+
|customer_id|event_type|amount|event_time         |error_reason       |is_valid|
+-----------+----------+------+-------------------+-------------------+--------+
|C001       |purchase  |100.0 |2026-01-01 10:00:00|NULL               |true    |
|C002       |refund    |50.0  |2026-01-01 11:00:00|NULL               |true    |
|C003       |purchase  |NULL  |2026-01-01 12:00:00|NULL               |true    |
|NULL       |purchase  |20.0  |2026-01-01 13:00:00|missing_customer_id|false   |
|C005       |unknown   |10.0  |2026-01-01 14:00:00|invalid_event_type |false   |
|C006       |purchase  |-5.0  |2026-01-01 15:00:00|negative_amount    |false   |
+-----------+----------+------+-------------------+-------------------+--------+



## Step 7 — Silver valid rows

In [8]:
silver_valid = (
    validated
    .filter(F.col("is_valid"))
    .select(
        "customer_id",
        "event_type",
        "amount",
        "event_time",
    )
)

silver_valid.show(truncate=False)

+-----------+----------+------+-------------------+
|customer_id|event_type|amount|event_time         |
+-----------+----------+------+-------------------+
|C001       |purchase  |100.0 |2026-01-01 10:00:00|
|C002       |refund    |50.0  |2026-01-01 11:00:00|
|C003       |purchase  |NULL  |2026-01-01 12:00:00|
+-----------+----------+------+-------------------+



## Step 8 — Quarantine rows

In [9]:
quarantine = (
    validated
    .filter(~F.col("is_valid"))
    .select(
        "customer_id",
        "event_type",
        "amount",
        "event_time",
        "error_reason",
    )
)

quarantine.show(truncate=False)

+-----------+----------+------+-------------------+-------------------+
|customer_id|event_type|amount|event_time         |error_reason       |
+-----------+----------+------+-------------------+-------------------+
|NULL       |purchase  |20.0  |2026-01-01 13:00:00|missing_customer_id|
|C005       |unknown   |10.0  |2026-01-01 14:00:00|invalid_event_type |
|C006       |purchase  |-5.0  |2026-01-01 15:00:00|negative_amount    |
+-----------+----------+------+-------------------+-------------------+



## Step 9 — Control totals

In [10]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("raw_rows", df.count()),
        ("silver_valid_rows", silver_valid.count()),
        ("quarantine_rows", quarantine.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+-----------------+-----+
|metric           |value|
+-----------------+-----+
|raw_rows         |6    |
|silver_valid_rows|3    |
|quarantine_rows  |3    |
+-----------------+-----+



## Final result

```text
Pydantic contract
      |
      v
Spark raw DataFrame
      |
      v
Spark validation
      |
      +--> Silver valid
      |
      +--> Quarantine
```

In [11]:
spark.stop()